# Tabular Q-learning on Pendulum-v1

Upload this file to GitHub to view it. To run it, open [Google Colab](https://colab.research.google.com/), select **GitHub** in the open dialog, and paste the notebook link (or upload the file). Run the cells from top to bottom. A CPU is sufficient; the first code cell installs the dependencies. This notebook is self-contained and does not require the `.py` files. You can also use it in Jupyter.

The goal is to swing the pendulum upright and keep it balanced. Rewards are negative: a return closer to zero is better. Discretization limits the accuracy of the controller.

In [ ]:
%pip install -q "gymnasium[classic-control]" numpy matplotlib
%matplotlib inline

## Parameters

Start with 2,000 episodes for an initial run; the original script uses 50,000. Increase `EPISODES` for longer training. Exploration stays fixed at 0.3, as in the script. For a quick check, use 10 episodes.

In [ ]:
import os
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")

import gymnasium as gym  # Gymnasium is the maintained successor to Gym.
import numpy as np


# Try changing these values to see how learning changes.
ANGLE_BINS = 40
VELOCITY_BINS = 40
TORQUES = np.array([-2.0, -1.0, 0.0, 1.0, 2.0], dtype=np.float32)
ALPHA = 0.1  # Learning rate: how much each new target changes a Q value.
GAMMA = 0.99  # Discount factor: how much future rewards matter.
SEED = 42

EPISODES = 2000
DEMO_EPISODES = 3

## Discretization and training

In [ ]:
def discretize(observation):
    """Convert [cos(angle), sin(angle), angular velocity] into two indices."""
    cosine, sine, velocity = observation
    angle = np.arctan2(sine, cosine)  # Zero means upright.
    # Map the angle to a circular grid: -pi and +pi represent the same pose.
    angle_index = int((angle + np.pi) / (2 * np.pi) * ANGLE_BINS) % ANGLE_BINS
    # Pendulum-v1 limits angular velocity to [-8, 8] rad/s.
    velocity_index = int((velocity + 8.0) / 16.0 * VELOCITY_BINS)
    velocity_index = min(max(velocity_index, 0), VELOCITY_BINS - 1)
    return angle_index, velocity_index


def train(episodes):
    rng = np.random.default_rng(SEED)
    # Q[angle bin, velocity bin, action] estimates discounted future reward.
    q = np.zeros((ANGLE_BINS, VELOCITY_BINS, len(TORQUES)))
    env = gym.make("Pendulum-v1")  # No rendering during training: much faster.
    recent_returns = []
    try:
        for episode in range(episodes):
            observation, _ = env.reset(seed=SEED if episode == 0 else None)
            state = discretize(observation)
            total_reward = 0.0
            # Start with random exploration; gradually favor learned actions.
            #epsilon = max(0.05, 1.0 - episode / max(1, 0.8 * episodes))
            epsilon = 0.3
            while True:
                if rng.random() < epsilon:
                    action = int(rng.integers(len(TORQUES)))
                else:
                    action = int(np.argmax(q[state]))

                # Gym expects a continuous torque array of shape (1,).
                observation, reward, terminated, truncated, _ = env.step(
                    np.array([TORQUES[action]], dtype=np.float32)
                )
                next_state = discretize(observation)

                # Q(s,a) <- Q(s,a) + alpha * [r + gamma * max Q(s',a') - Q(s,a)]
                # Bootstrap at time limits (truncated), but not true terminal states.
                future_value = 0.0 if terminated else np.max(q[next_state])
                target = reward + GAMMA * future_value
                q[state + (action,)] += ALPHA * (target - q[state + (action,)])

                state = next_state
                total_reward += reward
                if terminated or truncated:
                    break

            recent_returns.append(total_reward)
            if (episode + 1) % 100 == 0 or episode == episodes - 1:
                print(f"Episode {episode + 1:5d}/{episodes} | "
                      f"mean return (last 100): {np.mean(recent_returns[-100:]):8.1f} | "
                      f"epsilon: {epsilon:.2f}")
    finally:
        env.close()
    return q, recent_returns

## Reward plot

In [ ]:
def plot_rewards(returns):
    """Plot training returns and a moving average to make the trend clearer."""
    import matplotlib.pyplot as plt

    episodes = np.arange(1, len(returns) + 1)
    # Use a shorter window when training for fewer than 100 episodes.
    window = min(100, len(returns))
    moving_average = np.convolve(returns, np.ones(window) / window, mode="valid")

    fig, ax = plt.subplots()
    ax.plot(episodes, returns, alpha=0.3, label="Episode return")
    # The first average belongs to the last episode of the first full window.
    ax.plot(episodes[window - 1:], moving_average,
            label=f"Moving average ({window} episodes)")
    ax.set_title("Pendulum Q-learning: training rewards")
    ax.set_xlabel("Episode")
    ax.set_ylabel("Total reward (higher is better)")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    plt.show()
    plt.close(fig)

## Start training

In [ ]:
assert isinstance(EPISODES, int) and EPISODES >= 1
q_table, returns = train(EPISODES)
plot_rewards(returns)

## Watch the pendulum

The policy selects the action with the highest Q value without exploration. HTML animations play directly in the notebook, without GUI windows or FFmpeg.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

def visualize(q, episodes=3):
    env = gym.make("Pendulum-v1", render_mode="rgb_array")
    try:
        for episode in range(episodes):
            observation, _ = env.reset(seed=SEED + 1000 + episode)
            frames = [env.render()]
            total_reward = 0.0
            step = 0
            while True:
                action = int(np.argmax(q[discretize(observation)]))
                observation, reward, terminated, truncated, _ = env.step(
                    np.array([TORQUES[action]], dtype=np.float32)
                )
                total_reward += reward
                step += 1
                if step % 2 == 0 or terminated or truncated:
                    frames.append(env.render())
                if terminated or truncated:
                    break
            print(f"Demo {episode + 1}: return = {total_reward:.1f}")
            fig, ax = plt.subplots(figsize=(4, 4))
            artist = ax.imshow(frames[0])
            ax.axis("off")
            def update(index):
                artist.set_data(frames[index])
                return (artist,)
            animation = FuncAnimation(
                fig, update, frames=len(frames),
                interval=2000 / env.metadata.get("render_fps", 30), blit=True
            )
            plt.close(fig)
            display(HTML(animation.to_jshtml()))
    finally:
        env.close()

visualize(q_table, DEMO_EPISODES)